In [ ]:
import pandas as pd
import datasets

In [ ]:
sciq = pd.read_csv('/content/sciq_clean.csv')
arc = pd.read_csv('/content/arc_clean.csv')
mmlu = pd.read_csv('/content/mmlu_clean.csv')

In [ ]:
combined_df = pd.concat([sciq, arc, mmlu], ignore_index=True)
display(combined_df.head())

In [ ]:
combined_df= combined_df.sample(frac=1, random_state=42).reset_index(drop=True)
display(combined_df.head())

In [ ]:
combined_df.shape

In [ ]:
combined_df.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

combined_df['dataset_source'].value_counts().plot.pie(autopct='%1.1f%%')
plt.xticks(rotation=90)
plt.title('Dataset Distribution by Source')
plt.xlabel('Dataset Source')
plt.ylabel('Count')
plt.show()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
sns.countplot(data=combined_df, x='answer_key', order=['A', 'B', 'C', 'D'], palette='viridis')
plt.title('Answer Distribution')
plt.xlabel('Correct Option')
plt.ylabel('Number of Questions')
plt.show()

In [ ]:
combined_df.to_csv('combined_data.csv')

In [ ]:
train_df = combined_df.sample(frac=0.8, random_state=42)
test_df = combined_df.drop(train_df.index)

In [ ]:
from datasets import Dataset
hf_train = Dataset.from_pandas(train_df)
hf_test = Dataset.from_pandas(test_df)
print("Hugging Face Train Dataset:")
display(hf_train[0])
print("\nHugging Face Test Dataset:")
display(hf_test[0])

In [ ]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import os
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-13.0/lib64'

In [ ]:
from google.colab import userdata
from huggingface_hub import login

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token)
print("Authenticated successfully!")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
model_id = "google/gemma-2b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
print("Downloading Gemma 2B")
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
print("Sucess! The engine is running")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from transformers import pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
prompt = """Question: What is the main function of red blood cells?
Answer:"""
result = generator(prompt, max_new_tokens=50)
print("\n--- AI TEST RESPONSE ---")
print(result[0]['generated_text'])

In [ ]:
correct_answers = 0
total_questions = 100
print(f"Testing baseline Gemma 2B on {total_questions} questions...")

In [ ]:
import re

In [ ]:
for i in range(total_questions):
    raw_text = hf_test[i]['text']
    question_only = raw_text.split("Answer:")[0] + "Answer:"
    question_text = f"Instruction: You are an expert science student taking a test. You must reply with ONLY a single letter: A, B, C, or D.\n\n{question_only.strip()}"
    correct_answer = hf_test[i]['answer_key'].strip().upper()
    inputs = tokenizer(question_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=2, pad_token_id=tokenizer.eos_token_id, temperature=0.0, do_sample=False)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    clean_text = decoded.split("Answer:")[-1].upper().replace("OPTION", "")
    match = re.search(r'[ABCD]', clean_text)
    last_letter = match.group(0) if match else "X"
    if last_letter == correct_answer:
        correct_answers += 1
    if (i + 1) % 10 == 0:
        print(f"Graded {i+1}/{total_questions} questions...")

In [ ]:
baseline_accuracy = (correct_answers / total_questions) * 100
print("BASELINE GEMMA 2B RESULTS")
print(f"\tCorrect : {correct_answers} / {total_questions}")
print(f"\tAccuracy: {baseline_accuracy:.2f}%")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json
path = "/content/drive/MyDrive/Colab Notebooks/start1.ipynb"
with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)
nb.get("metadata", {}).pop("widgets", None)
for cell in nb.get("cells", []):
    cell.get("metadata", {}).pop("widgets", None)
with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)
print("Fixed notebook metadata!")


In [ ]:
# import json
# path = "/content/drive/MyDrive/Colab Notebooks/start1.ipynb"

# try:
#     with open(path, "r", encoding="utf-8") as f:
#         nb = json.load(f)
#     if "metadata" in nb and "widgets" in nb["metadata"]:
#         del nb["metadata"]["widgets"]
#         print("✅ Cleaned top-level metadata.")

#     cells_fixed = 0
#     for cell in nb.get("cells", []):
#         if "metadata" in cell and "widgets" in cell["metadata"]:
#             del cell["metadata"]["widgets"]
#             cells_fixed += 1

#         if "outputs" in cell:
#             clean_outputs = []
#             for out in cell["outputs"]:
#                 if "data" in out and "application/vnd.jupyter.widget-view+json" in out["data"]:
#                     cells_fixed += 1
#                     continue
#                 clean_outputs.append(out)
#             cell["outputs"] = clean_outputs
#     with open(path, "w", encoding="utf-8") as f:
#         json.dump(nb, f, ensure_ascii=False, indent=1)

#     print(f"✅ Deep clean complete! Removed {cells_fixed} hidden widget bugs. You are clear to upload.")

# except FileNotFoundError:
#     print("❌ ERROR: Could not find the file. Check the path!")